# Practical 6 — Microloan Lending Data Analysis

## Objective
Read lendingdata.csv, count object columns and missing values, identify likely redundant columns, calculate loan_amount Q3, repayment_interval percentages after dropping missing values, and minimum agriculture-sector loan.

**How to use:** Put the required CSV file(s) in the `data/` folder, update the filename/path if needed, and run cells from top to bottom. Where a dataset is unavailable, the notebook reports that clearly rather than inventing results.

## Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
csv_path = DATA_DIR / "lendingdata.csv"
if not csv_path.exists():
    raise FileNotFoundError("Place lendingdata.csv in the data/ folder.")
data = pd.read_csv(csv_path)
display(data.head())

In [ ]:
print("Object dtype column count:", int(data.select_dtypes(include=["object"]).shape[1]))
print("Total missing values:", int(data.isna().sum().sum()))
print("Missing values by column:")
display(data.isna().sum().loc[lambda s: s > 0].to_frame("missing_count"))

## Identify potentially redundant columns

In [ ]:
# Exact duplicate columns are objectively redundant. Other columns such as country and country_code
# may overlap semantically, but should only be dropped after confirming the intended analysis.
duplicate_columns = []
cols = list(data.columns)
for i, col in enumerate(cols):
    for other in cols[:i]:
        if data[col].equals(data[other]):
            duplicate_columns.append(col)
            break
print("Exact duplicate columns:", sorted(set(duplicate_columns)))
print("Potential semantic overlap to review manually: country and country_code; inspect before dropping.")
clean_data = data.drop(columns=sorted(set(duplicate_columns)))

In [ ]:
loan_col = "loan_amount"
if loan_col not in data.columns:
    raise KeyError(f"Expected loan_amount column. Available: {list(data.columns)}")
loan_amount = pd.to_numeric(data[loan_col], errors="coerce")
print("Third quartile of loan_amount:", loan_amount.quantile(0.75))

In [ ]:
repay_col = "repayment_interval"
if repay_col not in data.columns:
    raise KeyError("Expected repayment_interval column.")
repay = data[repay_col].dropna()
percentage_split = (repay.value_counts(normalize=True) * 100).rename("percentage").to_frame()
display(percentage_split.round(2))

In [ ]:
sector_col = "sector"
if sector_col not in data.columns:
    raise KeyError("Expected sector column.")
agri = data.loc[data[sector_col].astype(str).str.strip().str.casefold().eq("agriculture"), loan_col]
agri = pd.to_numeric(agri, errors="coerce").dropna()
if agri.empty:
    print("No valid loan_amount values found for sector='Agriculture'. Check category spelling/capitalization.")
else:
    print("Minimum loan amount in Agriculture sector:", agri.min())

## Conclusion

Use the printed output for the six questions. The PDF describes the variables but does not provide lendingdata.csv, so dataset-specific values must be calculated after supplying it.